In [85]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from sklearn.preprocessing import MinMaxScaler

In [86]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [87]:
# Need to choose patient_id from OUS_D2 in response_OUS
data = list(OUS_D2['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D2, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS', 'LRC', 'event_LRC'])]

In [88]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [89]:
# Check null values in D2 
clinical_train.isnull().sum().sum()

0

In [90]:
""" Takes too long time 
# Check collinearity for OUS data 
df = clinical_train

# Create a correlation matrix
correlation_matrix = df.corr()

plt.figure(figsize=(15, 12))

# Plot a heatmap
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.show()
""" 

" Takes too long time \n# Check collinearity for OUS data \ndf = clinical_train\n\n# Create a correlation matrix\ncorrelation_matrix = df.corr()\n\nplt.figure(figsize=(15, 12))\n\n# Plot a heatmap\nsns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)\nplt.show()\n"

## Test dataset: MAASTRO 

In [91]:
(MAASTRO_D2['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [92]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [93]:
# need to choose patient_id from MAASTRO_D2 in response_MAASTRO
data = list(MAASTRO_D2['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [94]:
# Merge MAASTRO_D2 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D2, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'OS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,shape_Elongation,shape_Flatness,shape_LeastAxisLength,shape_MajorAxisLength,shape_Maximum2DDiameterColumn,shape_Maximum2DDiameterRow,shape_Maximum2DDiameterSlice,shape_Maximum3DDiameter,shape_MeshVolume,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,DFS_event
0,1,0.765178,0.610062,30.504395,50.002093,52.886671,58.000000,47.759816,58.864251,38067.208333,...,0.028808,0.837387,0.000026,0.001705,0.122209,0.000026,0.008291,0.000341,62.43,0.0
1,2,0.776540,0.504616,21.069386,41.753334,44.922155,44.294469,44.147480,48.723711,17870.791667,...,0.049615,0.806842,0.000167,0.002958,0.128976,0.000167,0.008148,0.000335,60.00,0.0
2,3,0.697164,0.478604,21.238275,44.375483,48.466483,45.891176,37.656341,48.969378,17426.500000,...,0.019514,0.830272,0.000057,0.001831,0.137282,0.000000,0.008641,0.000229,8.83,1.0
3,4,0.574636,0.446059,20.570459,46.115989,42.544095,47.507894,33.837849,55.226805,12463.791667,...,0.047155,0.760949,0.000000,0.003597,0.171595,0.000080,0.013187,0.000240,19.73,1.0
4,6,0.633419,0.480378,26.130155,54.394967,66.030296,56.320511,45.398238,67.089492,28300.125000,...,0.033990,0.824865,0.000000,0.002116,0.128134,0.000035,0.009379,0.000529,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,0.882411,0.577884,19.774388,34.218615,41.785165,40.261644,35.805028,43.520110,12822.458333,...,0.017489,0.834590,0.000000,0.001321,0.137194,0.000078,0.008706,0.000078,13.27,1.0
95,111,0.535802,0.455642,23.259099,51.046869,51.264022,51.264022,33.600595,52.440442,18368.291667,...,0.029979,0.821431,0.000000,0.000543,0.137620,0.000054,0.008907,0.000489,85.87,1.0
96,112,0.716610,0.631485,31.838169,50.417953,46.324939,56.035703,57.070132,57.671483,35384.541667,...,0.017354,0.859957,0.000000,0.000988,0.115099,0.000028,0.005700,0.000141,42.87,0.0
97,113,0.665145,0.628338,28.213278,44.901412,50.289164,50.596443,35.777088,51.478151,26180.208333,...,0.025399,0.847908,0.000000,0.001487,0.117654,0.000000,0.005759,0.000038,58.93,0.0


In [95]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [96]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,shape_Elongation,shape_Flatness,shape_LeastAxisLength,shape_MajorAxisLength,shape_Maximum2DDiameterColumn,shape_Maximum2DDiameterRow,shape_Maximum2DDiameterSlice,shape_Maximum3DDiameter,shape_MeshVolume,shape_MinorAxisLength,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,DFS_event


In [97]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS'])]

# y 
y = clinical_train.loc[:, ['DFS', 'event_DFS']]

# Set lower, upper time point and times for IBS calculation later 
lower, upper = np.percentile(y['DFS'], [10, 90])
times = np.arange(lower, upper)

In [98]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [99]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 374)
y_train:  (139,)


In [100]:
clinical_test.rename(columns = {'DFS_event' : 'event_DFS'}, inplace = True)

In [101]:
# X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'event_DFS'])]

# y y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['DFS', 'event_DFS']]
lower, upper = np.percentile(y_MAASTRO['DFS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_DFS'], y_MAASTRO['DFS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 376)

# Feature Selection

In [102]:
# Choose features from the result of Cox PLSR in R
plsr = [
"shape_MajorAxisLength",
"shape_Elongation",
"glszm_LargeAreaLowGrayLevelEmphasis_CT_c16",
"shape_Sphericity",
"shape_MinorAxisLength",
"shape_SurfaceVolumeRatio",
"LBP_201_PET",
"gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_b2",
"shape_Flatness",
"glszm_GrayLevelNonUniformityNormalized_PET_c04"
]

In [103]:
X_plsr = X.loc[:, plsr]
X_new = X_plsr.copy()

In [104]:
# Selecct the columns from X_MAASTRO
MAASTRO_new = X_MAASTRO.loc[:, plsr]

# Standardization

In [105]:
# Copy the original X for later 
original_X = X.copy()

In [106]:
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Standardize X_new, the new data with the selected features only 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
scaler = MinMaxScaler() 
X_new_numeric_columns = X_new_numeric.columns
X_new_numeric_index = X_new_numeric.index 
X_new_numeric_std = scaler.fit_transform(X_new_numeric)
X_new_numeric_std = pd.DataFrame(X_new_numeric_std,
                                 columns=X_new_numeric_columns, 
                                 index=X_new_numeric_index)
X_new_std = pd.concat([X_new_numeric_std, X_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]

In [107]:
# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new_std = MAASTRO_new_std[MAASTRO_new.columns]

In [108]:
X_new

,shape_MajorAxisLength,shape_Elongation,glszm_LargeAreaLowGrayLevelEmphasis_CT_c16,shape_Sphericity,shape_MinorAxisLength,shape_SurfaceVolumeRatio,LBP_201_PET,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_b2,shape_Flatness,glszm_GrayLevelNonUniformityNormalized_PET_c04
0,42.073251,0.600926,7442.066260,0.761164,25.282894,0.251218,0.000062,0.000959,0.535140,0.140496
1,24.613845,0.841579,238.870676,0.697049,20.714498,0.489853,0.000349,0.002776,0.367109,0.180556
2,48.030294,0.772821,12957.157420,0.565792,37.118833,0.278467,0.000000,0.001179,0.597785,0.251029
3,25.589900,0.847727,233.632355,0.684364,21.693241,0.474018,0.000000,0.002748,0.405730,0.190083
4,34.684750,0.831483,121.921768,0.503142,28.839789,0.563135,0.000399,0.002309,0.442406,0.160494
...,...,...,...,...,...,...,...,...,...,...
134,33.069705,0.680294,1528.464999,0.742102,22.497115,0.322021,0.000000,0.001347,0.523608,0.285714
135,41.043692,0.758193,41182.958565,0.722918,31.119023,0.227705,0.000079,0.000850,0.735524,0.200000
136,36.618802,0.770113,3964.057173,0.652963,28.200620,0.298398,0.000000,0.001111,0.648063,0.173469
137,45.870392,0.628897,5243.112516,0.724255,28.847736,0.252893,0.000054,0.000935,0.492193,0.263889


In [109]:
X_new_std

,shape_MajorAxisLength,shape_Elongation,glszm_LargeAreaLowGrayLevelEmphasis_CT_c16,shape_Sphericity,shape_MinorAxisLength,shape_SurfaceVolumeRatio,LBP_201_PET,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_b2,shape_Flatness,glszm_GrayLevelNonUniformityNormalized_PET_c04
0,0.350904,0.461898,0.005519,0.828413,0.236549,0.263957,0.088609,0.138260,0.499075,0.031357
1,0.123069,0.829873,0.000158,0.645006,0.167245,0.707641,0.501821,0.522853,0.220565,0.112421
2,0.428640,0.724738,0.009624,0.269537,0.416103,0.314620,0.000000,0.184826,0.602909,0.255029
3,0.135806,0.839273,0.000154,0.608721,0.182093,0.678200,0.000000,0.516826,0.284579,0.131700
4,0.254489,0.814436,0.000071,0.090323,0.290508,0.843892,0.572624,0.424002,0.345369,0.071825
...,...,...,...,...,...,...,...,...,...,...
134,0.233413,0.583257,0.001118,0.773882,0.194288,0.395598,0.000000,0.220460,0.479962,0.325218
135,0.337469,0.702369,0.030632,0.719008,0.325084,0.220239,0.113114,0.115152,0.831212,0.151769
136,0.279727,0.720597,0.002931,0.518897,0.280811,0.351677,0.000000,0.170483,0.686245,0.098082
137,0.400455,0.504667,0.003883,0.722831,0.290628,0.267071,0.077871,0.133243,0.427891,0.281053


In [110]:
MAASTRO_new 

,shape_MajorAxisLength,shape_Elongation,glszm_LargeAreaLowGrayLevelEmphasis_CT_c16,shape_Sphericity,shape_MinorAxisLength,shape_SurfaceVolumeRatio,LBP_201_PET,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_b2,shape_Flatness,glszm_GrayLevelNonUniformityNormalized_PET_c04
0,50.002093,0.765178,9404.025346,0.668072,38.260495,0.215184,0.000026,0.000768,0.610062,0.232422
1,41.753334,0.776540,16468.863361,0.669961,32.423122,0.276092,0.000167,0.001213,0.504616,0.339506
2,44.375483,0.697164,3829.811759,0.624081,30.936983,0.298887,0.000000,0.001176,0.478604,0.171875
3,46.115989,0.574636,2212.056593,0.577624,26.499895,0.361096,0.000080,0.001192,0.446059,0.166667
4,54.394967,0.633419,20185.616389,0.630933,34.454789,0.251519,0.000035,0.000766,0.480378,0.301020
...,...,...,...,...,...,...,...,...,...,...
94,34.218615,0.882411,18006.593217,0.671754,30.194871,0.307574,0.000078,0.001137,0.577884,0.202216
95,51.046869,0.535802,1303.136099,0.632189,27.351039,0.289922,0.000054,0.000962,0.455642,0.157025
96,50.417953,0.716610,8928.353080,0.645548,36.130031,0.228184,0.000028,0.000621,0.631485,0.460317
97,44.901412,0.665145,13835.684015,0.727488,29.865942,0.223872,0.000000,0.000947,0.628338,0.314879


In [111]:
MAASTRO_new_std

,shape_MajorAxisLength,shape_Elongation,glszm_LargeAreaLowGrayLevelEmphasis_CT_c16,shape_Sphericity,shape_MinorAxisLength,shape_SurfaceVolumeRatio,LBP_201_PET,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_b2,shape_Flatness,glszm_GrayLevelNonUniformityNormalized_PET_c04
0,0.454371,0.713050,0.006979,0.562115,0.433423,0.196961,0.037694,0.097918,0.623259,0.217377
1,0.346729,0.730423,0.012238,0.567520,0.344868,0.310203,0.240548,0.192033,0.448482,0.434071
2,0.380947,0.609052,0.002831,0.436277,0.322323,0.352585,0.000000,0.184231,0.405367,0.094855
3,0.403660,0.421699,0.001627,0.303383,0.255011,0.468248,0.114827,0.187680,0.351425,0.084316
4,0.511695,0.511582,0.015004,0.455879,0.375689,0.264517,0.050658,0.097424,0.408308,0.356191
...,...,...,...,...,...,...,...,...,...,...
94,0.248406,0.892307,0.013382,0.572648,0.311065,0.368738,0.111676,0.175997,0.569924,0.156253
95,0.468005,0.362320,0.000950,0.459472,0.267923,0.335917,0.078027,0.138832,0.367308,0.064805
96,0.459798,0.638788,0.006625,0.497683,0.401103,0.221130,0.040540,0.066663,0.658767,0.678542
97,0.387810,0.560093,0.010278,0.732079,0.306075,0.213114,0.000000,0.135787,0.653551,0.384235


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [112]:
# Setting the y format for skf below  
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-15 13:33:21,120] A new study created in memory with name: no-name-51b7705f-4a87-4318-8792-b9b03eabda1e


  0%|          | 0/1 [00:00<?, ?it/s]

[I 2024-04-15 13:33:21,408] A new study created in memory with name: no-name-14f97856-9afa-4a2f-bd7c-99a26d4822dd


Fold 1 C-index: 0.5577689243027888
Fold 2 C-index: 0.7441860465116279
Fold 3 C-index: 0.8042553191489362
Fold 4 C-index: 0.6958174904942965
Fold 5 C-index: 0.7124463519313304
[I 2024-04-15 13:33:21,402] Trial 0 finished with value: 0.7028948264777959 and parameters: {}. Best is trial 0 with value: 0.7028948264777959.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7028948264777959], datetime_start=datetime.datetime(2024, 4, 15, 13, 33, 21, 184048), datetime_complete=datetime.datetime(2024, 4, 15, 13, 33, 21, 401117), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7028948264777959


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.26188816761455536
Fold 2 IBS: 0.17457592134628114
Fold 3 IBS: 0.15644372418899147
Fold 4 IBS: 0.2105625191550176
Fold 5 IBS: 0.191398831336509
[I 2024-04-15 13:33:21,615] Trial 0 finished with value: 0.1989738327282709 and parameters: {}. Best is trial 0 with value: 0.1989738327282709.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.1989738327282709], datetime_start=datetime.datetime(2024, 4, 15, 13, 33, 21, 433092), datetime_complete=datetime.datetime(2024, 4, 15, 13, 33, 21, 615521), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.1989738327282709


In [113]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [114]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.703
train_ibs:  0.199


#### Test

In [115]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [116]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.528
IBS score: 0.309


In [117]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [118]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis

#### Train

In [119]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-15 13:33:21,732] A new study created in memory with name: no-name-598da623-6695-45e5-ad25-01251f054d69


  0%|          | 0/1 [00:00<?, ?it/s]

[I 2024-04-15 13:33:21,826] A new study created in memory with name: no-name-77ad6b98-1049-4390-9b38-fc99124b28f1


Fold 1 C-index: 0.6115537848605578
Fold 2 C-index: 0.6841085271317829
Fold 3 C-index: 0.5319148936170213
Fold 4 C-index: 0.6806083650190115
Fold 5 C-index: 0.6502145922746781
[I 2024-04-15 13:33:21,820] Trial 0 finished with value: 0.6316800325806103 and parameters: {}. Best is trial 0 with value: 0.6316800325806103.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6316800325806103], datetime_start=datetime.datetime(2024, 4, 15, 13, 33, 21, 752009), datetime_complete=datetime.datetime(2024, 4, 15, 13, 33, 21, 820189), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6316800325806103


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.24724709891124252
Fold 2 IBS: 0.23203988190746228
Fold 3 IBS: 0.2289818679949847
Fold 4 IBS: 0.24197476918755945
Fold 5 IBS: 0.22939559123303355
[I 2024-04-15 13:33:21,938] Trial 0 finished with value: 0.2359278418468565 and parameters: {}. Best is trial 0 with value: 0.2359278418468565.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.2359278418468565], datetime_start=datetime.datetime(2024, 4, 15, 13, 33, 21, 853074), datetime_complete=datetime.datetime(2024, 4, 15, 13, 33, 21, 938630), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.2359278418468565


In [120]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [121]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.632
train_ibs:  0.236


#### Test

In [122]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [123]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.531


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.229


In [124]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [125]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-15 13:33:22,041] A new study created in memory with name: no-name-9896c6a1-30e0-4d09-9062-e63afc143e02


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5577689243027888
Fold 2 C-index: 0.7403100775193798
Fold 3 C-index: 0.8


[I 2024-04-15 13:33:22,268] A new study created in memory with name: no-name-1a5e2a5f-d66a-42f7-8887-a5c97ecd9094


Fold 4 C-index: 0.6996197718631179
Fold 5 C-index: 0.7081545064377682
[I 2024-04-15 13:33:22,263] Trial 0 finished with value: 0.7011706560246108 and parameters: {}. Best is trial 0 with value: 0.7011706560246108.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7011706560246108], datetime_start=datetime.datetime(2024, 4, 15, 13, 33, 22, 62023), datetime_complete=datetime.datetime(2024, 4, 15, 13, 33, 22, 263580), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7011706560246108


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.2618548971908547
Fold 2 IBS: 0.17535907857190686
Fold 3 IBS: 0.15772602324118667
Fold 4 IBS: 0.21093897607088266
Fold 5 IBS: 0.19001858270911773
[I 2024-04-15 13:33:22,515] Trial 0 finished with value: 0.19917951155678973 and parameters: {}. Best is trial 0 with value: 0.19917951155678973.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.19917951155678973], datetime_start=datetime.datetime(2024, 4, 15, 13, 33, 22, 293805), datetime_complete=datetime.datetime(2024, 4, 15, 13, 33, 22, 515423), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.19917951155678973


In [126]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [127]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.701
train_ibs:  0.199


#### Test

In [128]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [129]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.528


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.308


In [130]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [131]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-15 13:33:22,673] A new study created in memory with name: no-name-6155cf04-86dc-42be-b6cc-540eb479f36e


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5577689243027888
Fold 2 C-index: 0.7403100775193798
Fold 3 C-index: 0.8085106382978723
Fold 4 C-index: 0.6996197718631179
Fold 5 C-index: 0.7081545064377682
[I 2024-04-15 13:33:22,899] Trial 0 finished with value: 0.7028727836841854 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.7028727836841854.
Fold 1 C-index: 0.5577689243027888
Fold 2 C-index: 0.7403100775193798
Fold 3 C-index: 0.8127659574468085
Fold 4 C-index: 0.6996197718631179
Fold 5 C-index: 0.7081545064377682
[I 2024-04-15 13:33:23,145] Trial 1 finished with value: 0.7037238475139727 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.7037238475139727.
Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.7403100775193798
Fold 3 C-index: 0.548936170212766
Fold 4 C-index: 0.6996197718631179
Fold 5 C-index: 0.7081545064377682
[I 2024-04-15 13:33:23,413] Trial 2 finished with value: 0.6517547028161681 and parameters: {'l1_ratio': 0.22692876841884668}. 

Fold 1 C-index: 0.5577689243027888
Fold 2 C-index: 0.7403100775193798
Fold 3 C-index: 0.8085106382978723
Fold 4 C-index: 0.6996197718631179
Fold 5 C-index: 0.7081545064377682
[I 2024-04-15 13:33:27,252] Trial 25 finished with value: 0.7028727836841854 and parameters: {'l1_ratio': 0.6016458784897931}. Best is trial 11 with value: 0.7045206602629766.
Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.7403100775193798
Fold 3 C-index: 0.548936170212766
Fold 4 C-index: 0.6996197718631179
Fold 5 C-index: 0.7081545064377682
[I 2024-04-15 13:33:27,421] Trial 26 finished with value: 0.6517547028161681 and parameters: {'l1_ratio': 0.24991176779209634}. Best is trial 11 with value: 0.7045206602629766.
Fold 1 C-index: 0.5577689243027888
Fold 2 C-index: 0.7403100775193798
Fold 3 C-index: 0.8127659574468085
Fold 4 C-index: 0.6996197718631179
Fold 5 C-index: 0.7081545064377682
[I 2024-04-15 13:33:27,600] Trial 27 finished with value: 0.7037238475139727 and parameters: {'l1_ratio': 0.350740277867186

Fold 1 C-index: 0.5577689243027888
Fold 2 C-index: 0.7403100775193798
Fold 3 C-index: 0.8127659574468085
Fold 4 C-index: 0.6996197718631179
Fold 5 C-index: 0.7081545064377682
[I 2024-04-15 13:33:31,072] Trial 49 finished with value: 0.7037238475139727 and parameters: {'l1_ratio': 0.31096634412376933}. Best is trial 11 with value: 0.7045206602629766.
Fold 1 C-index: 0.6055776892430279
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.5531914893617021
Fold 4 C-index: 0.6806083650190115
Fold 5 C-index: 0.6824034334763949
[I 2024-04-15 13:33:31,129] Trial 50 finished with value: 0.644666272939407 and parameters: {'l1_ratio': 0.12981789694574747}. Best is trial 11 with value: 0.7045206602629766.
Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.7403100775193798
Fold 3 C-index: 0.8127659574468085
Fold 4 C-index: 0.6996197718631179
Fold 5 C-index: 0.7081545064377682
[I 2024-04-15 13:33:31,347] Trial 51 finished with value: 0.7045206602629766 and parameters: {'l1_ratio': 0.25794490371281

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.26112506117169937
Fold 2 IBS: 0.17539322866078413
Fold 3 IBS: 0.15749720573738968
Fold 4 IBS: 0.210901523737872
Fold 5 IBS: 0.18993209550030718
[I 2024-04-15 13:33:40,583] Trial 0 finished with value: 0.19896982296161048 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.19896982296161048.
Fold 1 IBS: 0.2608554376088747
Fold 2 IBS: 0.17516304746695047
Fold 3 IBS: 0.15722660100172273
Fold 4 IBS: 0.2108811839542744
Fold 5 IBS: 0.18996281056969944
[I 2024-04-15 13:33:40,794] Trial 1 finished with value: 0.19881781612030436 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.19881781612030436.
Fold 1 IBS: 0.2608411182829978
Fold 2 IBS: 0.17510633367570527
Fold 3 IBS: 0.22888934703897795
Fold 4 IBS: 0.2108739930332697
Fold 5 IBS: 0.18995494728414902
[I 2024-04-15 13:33:40,978] Trial 2 finished with value: 0.21313314786301993 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 1 with value: 0.198817816120304

Fold 3 IBS: 0.2288900588406883
Fold 4 IBS: 0.21087561155991283
Fold 5 IBS: 0.18995391936920947
[I 2024-04-15 13:33:46,518] Trial 26 finished with value: 0.21313910170616182 and parameters: {'l1_ratio': 0.24616747074631654}. Best is trial 11 with value: 0.19881359579903582.
Fold 1 IBS: 0.2609127279468582
Fold 2 IBS: 0.17520014353853527
Fold 3 IBS: 0.15722538997452715
Fold 4 IBS: 0.2108677192427991
Fold 5 IBS: 0.18998633402187168
[I 2024-04-15 13:33:46,737] Trial 27 finished with value: 0.1988384629449183 and parameters: {'l1_ratio': 0.3397747316535139}. Best is trial 11 with value: 0.19881359579903582.
Fold 1 IBS: 0.2460590185192229
Fold 2 IBS: 0.23000317310124618
Fold 3 IBS: 0.22893198666557424
Fold 4 IBS: 0.24022330373865497
Fold 5 IBS: 0.2279802197234489
[I 2024-04-15 13:33:46,806] Trial 28 finished with value: 0.23463954034962944 and parameters: {'l1_ratio': 0.07729981217319248}. Best is trial 11 with value: 0.19881359579903582.
Fold 1 IBS: 0.26151157669893454
Fold 2 IBS: 0.17534939

Fold 1 IBS: 0.2609029813562119
Fold 2 IBS: 0.17511565022015008
Fold 3 IBS: 0.2288894029427478
Fold 4 IBS: 0.23697131495704718
Fold 5 IBS: 0.18993427092693696
[I 2024-04-15 13:33:51,769] Trial 52 finished with value: 0.2183627240806188 and parameters: {'l1_ratio': 0.2225918073443785}. Best is trial 11 with value: 0.19881359579903582.
Fold 1 IBS: 0.2609141656879759
Fold 2 IBS: 0.17517254220289363
Fold 3 IBS: 0.15724486047677114
Fold 4 IBS: 0.2108884783517394
Fold 5 IBS: 0.18993653853858755
[I 2024-04-15 13:33:51,982] Trial 53 finished with value: 0.19883131705159354 and parameters: {'l1_ratio': 0.2807107457107132}. Best is trial 11 with value: 0.19881359579903582.
Fold 1 IBS: 0.2609360704247251
Fold 2 IBS: 0.17523747495901695
Fold 3 IBS: 0.1573298007379569
Fold 4 IBS: 0.21091565587339614
Fold 5 IBS: 0.18998586173744106
[I 2024-04-15 13:33:52,254] Trial 54 finished with value: 0.19888097274650723 and parameters: {'l1_ratio': 0.42607274095914727}. Best is trial 11 with value: 0.19881359579

Fold 5 IBS: 0.22581807649536825
[I 2024-04-15 13:33:56,563] Trial 77 finished with value: 0.23601823544951367 and parameters: {'l1_ratio': 0.19328346899350535}. Best is trial 75 with value: 0.19880594318930178.
Fold 1 IBS: 0.2608997001884041
Fold 2 IBS: 0.1751744113251068
Fold 3 IBS: 0.15721450525553415
Fold 4 IBS: 0.21089631616353507
Fold 5 IBS: 0.18998366753711324
[I 2024-04-15 13:33:56,776] Trial 78 finished with value: 0.19883372009393868 and parameters: {'l1_ratio': 0.3160690426163704}. Best is trial 75 with value: 0.19880594318930178.
Fold 1 IBS: 0.26179949727647756
Fold 2 IBS: 0.1753873088968012
Fold 3 IBS: 0.15775663412474292
Fold 4 IBS: 0.21094749759571413
Fold 5 IBS: 0.18997802322605103
[I 2024-04-15 13:33:56,946] Trial 79 finished with value: 0.19917379222395737 and parameters: {'l1_ratio': 0.9846333768922103}. Best is trial 75 with value: 0.19880594318930178.
Fold 1 IBS: 0.2608099364957766
Fold 2 IBS: 0.22804261450619429
Fold 3 IBS: 0.22890091367027132
Fold 4 IBS: 0.2385543

In [132]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [133]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.705
train_ibs:  0.199


#### Test

In [134]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [135]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.2694535558091148)

test_cindex : 0.528


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.2660168592917107)

test_ibs:  0.308


In [136]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [137]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-15 13:34:00,281] A new study created in memory with name: no-name-f111fe4a-899d-4385-8aef-527663b79904


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6573705179282868
Fold 2 C-index: 0.8023255813953488
Fold 3 C-index: 0.6297872340425532
Fold 4 C-index: 0.7414448669201521
Fold 5 C-index: 0.6931330472103004
[I 2024-04-15 13:34:01,838] Trial 0 finished with value: 0.7048122494993283 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.7048122494993283.
Fold 1 C-index: 0.6055776892430279
Fold 2 C-index: 0.7674418604651163
Fold 3 C-index: 0.6553191489361702
Fold 4 C-index: 0.7604562737642585
Fold 5 C-index: 0.7339055793991416
[I 2024-04-15 13:34:02,880] Trial 1 finished with value: 0.7045401103615428 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 

Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.7868217054263565
Fold 3 C-index: 0.8085106382978723
Fold 4 C-index: 0.8365019011406845
Fold 5 C-index: 0.8197424892703863
[I 2024-04-15 13:34:17,511] Trial 16 finished with value: 0.7738213229226775 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 15, 'min_samples_leaf': 1, 'max_depth': 7, 'n_estimators': 122, 'oob_score': True, 'max_samples': 0.804668454581114, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.17004822192434552, 'warm_start': True}. Best is trial 14 with value: 0.7773044783671873.
Fold 1 C-index: 0.6414342629482072
Fold 2 C-index: 0.7906976744186046
Fold 3 C-index: 0.8085106382978723
Fold 4 C-index: 0.8174904942965779
Fold 5 C-index: 0.8025751072961373
[I 2024-04-15 13:34:17,848] Trial 17 finished with value: 0.77214163545148 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 16, 'min_samples_leaf': 10, 'max_depth': 8, 'n_estimators': 147, 'oob_score': True, 'max_samples': 0.8699768148803853,

Fold 1 C-index: 0.5816733067729084
Fold 2 C-index: 0.8294573643410853
Fold 3 C-index: 0.8936170212765957
Fold 4 C-index: 0.8821292775665399
Fold 5 C-index: 0.8583690987124464
[I 2024-04-15 13:34:24,813] Trial 31 finished with value: 0.8090492137339151 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 8, 'min_samples_leaf': 5, 'max_depth': 4, 'n_estimators': 307, 'oob_score': True, 'max_samples': 0.9009241102826697, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.05767674949496584, 'warm_start': True}. Best is trial 30 with value: 0.8187091599688708.
Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.8410852713178295
Fold 3 C-index: 0.8893617021276595
Fold 4 C-index: 0.8745247148288974
Fold 5 C-index: 0.871244635193133
[I 2024-04-15 13:34:25,461] Trial 32 finished with value: 0.8155619897931056 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 5, 'n_estimators': 307, 'oob_score': True, 'max_samples': 0.9444789533658244, '

...

Fold 1 IBS: 0.22871119834201115
Fold 2 IBS: 0.18629984266823443
Fold 3 IBS: 0.2136333562391605
Fold 4 IBS: 0.20508306074253652
Fold 5 IBS: 0.20460862507387245
[I 2024-04-15 13:36:21,372] Trial 32 finished with value: 0.20766721661316295 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 7, 'min_samples_leaf': 15, 'max_depth': 3, 'n_estimators': 310, 'oob_score': False, 'max_samples': 0.8321993039875358, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.16244513867075333}. Best is trial 1 with value: 0.20613771601435493.
Fold 1 IBS: 0.2450153519251981
Fold 2 IBS: 0.19177067560329883
Fold 3 IBS: 0.2050380118625506
Fold 4 IBS: 0.19783492315190962
Fold 5 IBS: 0.20502773397972165
[I 2024-04-15 13:36:22,848] Trial 33 finished with value: 0.20893733930453579 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 5, 'min_samples_leaf': 7, 'max_depth': 4, 'n_estimators': 224, 'oob_score': False, 'max_samples': 0.9881033887741655, 'max_features': 'log2', 'min_weight_fraction_lea

Fold 1 IBS: 0.2464013758053453
Fold 2 IBS: 0.23207408714903174
Fold 3 IBS: 0.22939892327646114
Fold 4 IBS: 0.2415689721009919
Fold 5 IBS: 0.23043319376810475
[I 2024-04-15 13:36:41,793] Trial 48 finished with value: 0.23597531041998696 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 7, 'n_estimators': 108, 'oob_score': True, 'max_samples': 0.6344615640154665, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.44962767788541247}. Best is trial 1 with value: 0.20613771601435493.
Fold 1 IBS: 0.24633584274807316
Fold 2 IBS: 0.23218539826865298
Fold 3 IBS: 0.22972627891694997
Fold 4 IBS: 0.2411567932158465
Fold 5 IBS: 0.2301175121753443
[I 2024-04-15 13:36:42,710] Trial 49 finished with value: 0.2359043650649734 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 13, 'min_samples_leaf': 20, 'max_depth': 2, 'n_estimators': 168, 'oob_score': False, 'max_samples': 0.2700643631363231, 'max_features': 'auto', 'min_weight_fraction_leaf'

Fold 1 IBS: 0.2263102371899488
Fold 2 IBS: 0.1935254758393429
Fold 3 IBS: 0.20737484396299696
Fold 4 IBS: 0.2086187676158567
Fold 5 IBS: 0.20696146438052143
[I 2024-04-15 13:36:54,923] Trial 64 finished with value: 0.2085581577977334 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 9, 'min_samples_leaf': 3, 'max_depth': 6, 'n_estimators': 252, 'oob_score': False, 'max_samples': 0.5041409232103772, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.1346031511543957}. Best is trial 52 with value: 0.2056012262727213.
Fold 1 IBS: 0.23692845343363148
Fold 2 IBS: 0.19041679735965694
Fold 3 IBS: 0.21008157321222845
Fold 4 IBS: 0.1987406581683996
Fold 5 IBS: 0.20283869466958301
[I 2024-04-15 13:36:56,947] Trial 65 finished with value: 0.2078012353686999 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 4, 'min_samples_leaf': 4, 'max_depth': 3, 'n_estimators': 340, 'oob_score': False, 'max_samples': 0.689407190096443, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.

Fold 1 IBS: 0.2300650150874554
Fold 2 IBS: 0.18010561006896428
Fold 3 IBS: 0.21441487116790292
Fold 4 IBS: 0.19757686931100543
Fold 5 IBS: 0.21275664692284824
[I 2024-04-15 13:37:14,024] Trial 81 finished with value: 0.20698380251163523 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 8, 'min_samples_leaf': 5, 'max_depth': 4, 'n_estimators': 70, 'oob_score': False, 'max_samples': 0.9023474857295949, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.16360107456547565}. Best is trial 52 with value: 0.2056012262727213.
Fold 1 IBS: 0.23002497302639052
Fold 2 IBS: 0.18368245768130084
Fold 3 IBS: 0.20926256560956186
Fold 4 IBS: 0.20438983840996378
Fold 5 IBS: 0.21178805819403893
[I 2024-04-15 13:37:14,482] Trial 82 finished with value: 0.20782957858425122 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 8, 'min_samples_leaf': 3, 'max_depth': 4, 'n_estimators': 66, 'oob_score': False, 'max_samples': 0.8041501696371655, 'max_features': 'log2', 'min_weight_fraction_leaf

Fold 2 IBS: 0.18415089303949564
Fold 3 IBS: 0.2084960116600641
Fold 4 IBS: 0.20343560492304824
Fold 5 IBS: 0.20851133716775203
[I 2024-04-15 13:37:20,984] Trial 97 finished with value: 0.20504804569085272 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 10, 'min_samples_leaf': 7, 'max_depth': 3, 'n_estimators': 95, 'oob_score': True, 'max_samples': 0.8475291147257535, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.20904972733074298}. Best is trial 97 with value: 0.20504804569085272.
Fold 1 IBS: 0.22137385042757987
Fold 2 IBS: 0.18840071162084657
Fold 3 IBS: 0.20838987304883333
Fold 4 IBS: 0.20228899120347085
Fold 5 IBS: 0.21075747268896797
[I 2024-04-15 13:37:21,610] Trial 98 finished with value: 0.20624217979793974 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 11, 'min_samples_leaf': 7, 'max_depth': 3, 'n_estimators': 95, 'oob_score': True, 'max_samples': 0.9260520619562637, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.2513983980188341}. Best is 

In [138]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [139]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.845
train_ibs:  0.205


#### Test

In [140]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [141]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=17, max_features='log2', max_leaf_nodes=14,
                     max_samples=0.8948299547976223, min_samples_leaf=2,
                     min_samples_split=7,
                     min_weight_fraction_leaf=0.03150292204046534,
                     n_estimators=213, oob_score=True, random_state=123,
                     warm_start=True)

test_cindex:  0.521


RandomSurvivalForest(max_depth=3, max_features='log2', max_leaf_nodes=10,
                     max_samples=0.8475291147257535, min_samples_leaf=7,
                     min_samples_split=2,
                     min_weight_fraction_leaf=0.20904972733074298,
                     n_estimators=95, oob_score=True, random_state=123)

test_ibs:  0.248


In [142]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [143]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [144]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-15 13:37:22,898] A new study created in memory with name: no-name-d1a10811-e460-401c-b2a4-d27cd06feaaa


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5737051792828686
Fold 2 C-index: 0.7635658914728682
Fold 3 C-index: 0.8
Fold 4 C-index: 0.7946768060836502
Fold 5 C-index: 0.8197424892703863
[I 2024-04-15 13:37:23,352] Trial 0 finished with value: 0.7503380732219547 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.7503380732219547.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 13:37:24,502] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531}. Best is tria

Fold 1 C-index: 0.5776892430278885
Fold 2 C-index: 0.7829457364341085
Fold 3 C-index: 0.7446808510638298
Fold 4 C-index: 0.7338403041825095
Fold 5 C-index: 0.7725321888412017
[I 2024-04-15 13:37:36,439] Trial 16 finished with value: 0.7223376647099076 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.6535355702555379, 'min_weight_fraction_leaf': 0.1906011147608997}. Best is trial 12 with value: 0.7665885346648663.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.7713178294573644
Fold 3 C-index: 0.7191489361702128
Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.7639484978540773
[I 2024-04-15 13:37:36,904] Trial 17 finished with value: 0.7176426303552769 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 10, 'min_samples_leaf': 8, 'max_depth': 10, 'n_estimators': 384, 'oob_score': False, 'warm_start': True, 'max_featu

Fold 1 C-index: 0.5976095617529881
Fold 2 C-index: 0.7713178294573644
Fold 3 C-index: 0.8127659574468085
Fold 4 C-index: 0.8022813688212928
Fold 5 C-index: 0.8154506437768241
[I 2024-04-15 13:37:44,410] Trial 31 finished with value: 0.7598850722510556 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 4, 'min_samples_leaf': 2, 'max_depth': 10, 'n_estimators': 458, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.9135692082875294, 'min_weight_fraction_leaf': 0.03048028383469985}. Best is trial 24 with value: 0.7703255353389682.
Fold 1 C-index: 0.5896414342629482
Fold 2 C-index: 0.7558139534883721
Fold 3 C-index: 0.8127659574468085
Fold 4 C-index: 0.779467680608365
Fold 5 C-index: 0.8111587982832618
[I 2024-04-15 13:37:44,940] Trial 32 finished with value: 0.7497695648179512 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 3, 'min_samples_leaf': 2, 'max_depth': 10, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_featur

...

Fold 1 IBS: 0.23852023407558698
Fold 2 IBS: 0.22085127198360574
Fold 3 IBS: 0.22246208066369713
Fold 4 IBS: 0.22864256467493396
Fold 5 IBS: 0.21968838126903262
[I 2024-04-15 13:40:06,412] Trial 30 finished with value: 0.22603290653337127 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 11, 'min_samples_leaf': 9, 'max_depth': 8, 'n_estimators': 457, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.3838693489032564, 'min_weight_fraction_leaf': 0.03868084640768137}. Best is trial 12 with value: 0.21246282745647482.
Fold 1 IBS: 0.23299946909800737
Fold 2 IBS: 0.20723625804859605
Fold 3 IBS: 0.21221943785442013
Fold 4 IBS: 0.2177017964639846
Fold 5 IBS: 0.20854416781687213
[I 2024-04-15 13:40:08,734] Trial 31 finished with value: 0.21574022585637603 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 6, 'max_depth': 13, 'n_estimators': 408, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 

Fold 1 IBS: 0.23535635522895523
Fold 2 IBS: 0.19710426247292917
Fold 3 IBS: 0.20557962919622366
Fold 4 IBS: 0.21419277453695665
Fold 5 IBS: 0.20885872177069337
[I 2024-04-15 13:40:32,350] Trial 45 finished with value: 0.2122183486411516 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 5, 'min_samples_leaf': 2, 'max_depth': 20, 'n_estimators': 226, 'oob_score': False, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.4468362954201356, 'min_weight_fraction_leaf': 0.022410036988178828}. Best is trial 35 with value: 0.21132948906228716.
Fold 1 IBS: 0.23697226819401304
Fold 2 IBS: 0.21729894679225342
Fold 3 IBS: 0.21924579151831378
Fold 4 IBS: 0.22736263277866023
Fold 5 IBS: 0.21648723912795567
[I 2024-04-15 13:40:33,725] Trial 46 finished with value: 0.2234733756822392 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 7, 'min_samples_leaf': 2, 'max_depth': 20, 'n_estimators': 246, 'oob_score': False, 'warm_start': False, 'max_features': 'log2', 'max_samples':

Fold 1 IBS: 0.24665906141024754
Fold 2 IBS: 0.23238673943832674
Fold 3 IBS: 0.22958317859156768
Fold 4 IBS: 0.24137956784118675
Fold 5 IBS: 0.23028136399594556
[I 2024-04-15 13:40:58,556] Trial 60 finished with value: 0.23605798225545485 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 11, 'n_estimators': 323, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.43142720229784703, 'min_weight_fraction_leaf': 0.48740241401755297}. Best is trial 56 with value: 0.20752977252523447.
Fold 1 IBS: 0.2321959013757636
Fold 2 IBS: 0.2007164455461997
Fold 3 IBS: 0.20770062793391472
Fold 4 IBS: 0.20602392627423452
Fold 5 IBS: 0.19819621614422855
[I 2024-04-15 13:40:59,691] Trial 61 finished with value: 0.20896662345486822 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 9, 'min_samples_leaf': 3, 'max_depth': 10, 'n_estimators': 193, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.3

Fold 1 IBS: 0.2355921510352545
Fold 2 IBS: 0.18988955633106214
Fold 3 IBS: 0.2175269995788695
Fold 4 IBS: 0.1993059384582704
Fold 5 IBS: 0.19950554004549564
[I 2024-04-15 13:41:10,696] Trial 75 finished with value: 0.20836403708979043 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 17, 'min_samples_leaf': 3, 'max_depth': 7, 'n_estimators': 62, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.6761462774417768, 'min_weight_fraction_leaf': 0.07610410535003291}. Best is trial 72 with value: 0.2074917391497911.
Fold 1 IBS: 0.2331646523531711
Fold 2 IBS: 0.19585793938278617
Fold 3 IBS: 0.21617307743244904
Fold 4 IBS: 0.20830690666506493
Fold 5 IBS: 0.2045693433216475
[I 2024-04-15 13:41:11,107] Trial 76 finished with value: 0.21161438383102374 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 18, 'min_samples_leaf': 4, 'max_depth': 7, 'n_estimators': 52, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.76838967

Fold 1 IBS: 0.2310593691941199
Fold 2 IBS: 0.19290862866485556
Fold 3 IBS: 0.2259889263099397
Fold 4 IBS: 0.20574406592712768
Fold 5 IBS: 0.2036881105655048
[I 2024-04-15 13:41:19,263] Trial 90 finished with value: 0.21187782013230955 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 18, 'min_samples_leaf': 2, 'max_depth': 3, 'n_estimators': 33, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.6403451257560147, 'min_weight_fraction_leaf': 0.1070974205302174}. Best is trial 72 with value: 0.2074917391497911.
Fold 1 IBS: 0.2401929507969826
Fold 2 IBS: 0.19402821024539957
Fold 3 IBS: 0.20959009756624364
Fold 4 IBS: 0.19925984870201024
Fold 5 IBS: 0.20019981509015003
[I 2024-04-15 13:41:20,169] Trial 91 finished with value: 0.2086541844801572 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 15, 'min_samples_leaf': 3, 'max_depth': 4, 'n_estimators': 131, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.69469242

In [145]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [146]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.837
train_ibs:  0.207


#### Test

In [147]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [148]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=18, max_features='auto', max_leaf_nodes=17,
                   max_samples=0.678838159065027, min_samples_leaf=1,
                   min_samples_split=2,
                   min_weight_fraction_leaf=0.011359003866057657,
                   n_estimators=362, oob_score=True, random_state=123,
                   warm_start=True)

C-index score: 0.509


ExtraSurvivalTrees(max_depth=7, max_features=None, max_leaf_nodes=15,
                   max_samples=0.7134910869727897, min_samples_split=2,
                   min_weight_fraction_leaf=0.0694822375534185, n_estimators=57,
                   random_state=123)

IBS: 0.259


In [149]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [150]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-15 13:41:26,880] A new study created in memory with name: no-name-75a771ca-4606-4acf-9ea6-209839ffefaf


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 13:41:38,597] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 13:41:45,776] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 13:44:53,825] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.6970416816836329.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 13:46:33,308] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 13:50:09,243] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.8502968404151126, 'learning_rate': 0.024443951259730985, 'dropout_rate': 0.2675273969344081, 'n_estimators': 441, 'criterion': 'squared_error', 'ccp_alpha': 2.0753749717266823, 'min_weight_fraction_leaf': 0.3503497788125578, 'max_features': 'auto', 'min_impurity_decrease': 7.237153572123947e-07, 'validation_fraction': 0.41222573804914475, 'min_samples_split': 16, 'max_leaf_nodes': 13, 'min_samples_leaf': 12, 'max_depth': 3}. Best is trial 9 with value: 0.6970416816836329.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 13:50:21,134] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.7670085127536703, 'learning_rate': 0.011828778593594373, 'dropout_rate': 0.4300954216773497, 'n_estimators': 330, 'criterion': 'friedman_mse', 'ccp_alpha':

...

Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.2293955930480925
[I 2024-04-15 15:39:36,984] Trial 55 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9027683411994925, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.13472529918097312, 'n_estimators': 381, 'criterion': 'squared_error', 'ccp_alpha': 1.3381569877935875, 'min_weight_fraction_leaf': 0.36938177041618503, 'max_features': 'auto', 'min_impurity_decrease': 1.1016843774656315e-07, 'validation_fraction': 0.898549324711475, 'min_samples_split': 3, 'max_leaf_nodes': 12, 'min_samples_leaf': 12, 'max_depth': 3}. Best is trial 53 with value: 0.2338206539157001.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-15 15:41:01,224] Trial 56 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.8324789538052517

Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-15 15:59:47,603] Trial 66 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9998769156045215, 'learning_rate': 0.010488456761899951, 'dropout_rate': 0.32975112844548055, 'n_estimators': 500, 'criterion': 'squared_error', 'ccp_alpha': 0.7134499429690735, 'min_weight_fraction_leaf': 0.19311178079767077, 'max_features': 'log2', 'min_impurity_decrease': 1.142816958470248e-06, 'validation_fraction': 0.916501041728546, 'min_samples_split': 19, 'max_leaf_nodes': 18, 'min_samples_leaf': 11, 'max_depth': 5}. Best is trial 53 with value: 0.2338206539157001.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-15 16:01:21,704] Trial 67 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9237031040797604

Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-15 16:16:04,441] Trial 77 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9372636724132823, 'learning_rate': 0.04128077713454106, 'dropout_rate': 0.2214138929394412, 'n_estimators': 441, 'criterion': 'squared_error', 'ccp_alpha': 0.5872217111247551, 'min_weight_fraction_leaf': 0.2869095730072229, 'max_features': 1, 'min_impurity_decrease': 1.637559261311236e-05, 'validation_fraction': 0.8634632458479912, 'min_samples_split': 19, 'max_leaf_nodes': 14, 'min_samples_leaf': 8, 'max_depth': 14}. Best is trial 53 with value: 0.2338206539157001.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-15 16:18:09,209] Trial 78 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7443996478445325, 'lear

Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-15 16:34:24,770] Trial 88 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.5839343488674055, 'learning_rate': 0.005616464421601462, 'dropout_rate': 0.35953503133089737, 'n_estimators': 446, 'criterion': 'squared_error', 'ccp_alpha': 0.3179664432907911, 'min_weight_fraction_leaf': 0.28749157637391265, 'max_features': 'auto', 'min_impurity_decrease': 3.335913971167387e-07, 'validation_fraction': 0.9542099213957375, 'min_samples_split': 6, 'max_leaf_nodes': 20, 'min_samples_leaf': 12, 'max_depth': 1}. Best is trial 53 with value: 0.2338206539157001.
Fold 1 IBS: 0.24665149166978298
Fold 2 IBS: 0.23100341911248753
Fold 3 IBS: 0.22874931175992053
Fold 4 IBS: 0.241449076384833
Fold 5 IBS: 0.22889139804833578
[I 2024-04-15 16:36:40,782] Trial 89 finished with value: 0.23534893939507198 and parameters: {'subsample': 0.9747597567605316,

Fold 2 IBS: 0.23172776185253358
Fold 3 IBS: 0.22878554072857857
Fold 4 IBS: 0.2416676429021851
Fold 5 IBS: 0.2290838777915168
[I 2024-04-15 16:50:38,203] Trial 99 finished with value: 0.23565647433985792 and parameters: {'subsample': 0.24286963781937687, 'learning_rate': 0.008804936390818944, 'dropout_rate': 0.14353784252843382, 'n_estimators': 498, 'criterion': 'squared_error', 'ccp_alpha': 0.004530258324353442, 'min_weight_fraction_leaf': 0.29607175965989646, 'max_features': 0.1, 'min_impurity_decrease': 1.9888370412270107e-06, 'validation_fraction': 0.9539243209909428, 'min_samples_split': 19, 'max_leaf_nodes': 20, 'min_samples_leaf': 13, 'max_depth': 2}. Best is trial 53 with value: 0.2338206539157001.


* Best trial for IBS: 
 FrozenTrial(number=53, state=TrialState.COMPLETE, values=[0.2338206539157001], datetime_start=datetime.datetime(2024, 4, 15, 15, 34, 28, 59632), datetime_complete=datetime.datetime(2024, 4, 15, 15, 36, 20, 582002), params={'subsample': 0.9101431135071837, 'l

In [151]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [152]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.711
train_ibs:  0.234


#### Test

In [153]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [154]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.02313127962403605,
                                 criterion='squared_error',
                                 dropout_rate=0.7413014948736192,
                                 learning_rate=0.03158550801216725,
                                 max_features='sqrt', max_leaf_nodes=20,
                                 min_impurity_decrease=1.9076132907505136e-05,
                                 min_samples_leaf=11, min_samples_split=18,
                                 min_weight_fraction_leaf=0.25025224628887144,
                                 n_estimators=449, random_state=123,
                                 subsample=0.5109023203910539,
                                 validation_fraction=0.8776959526028103)

C-index score: 0.517


GradientBoostingSurvivalAnalysis(ccp_alpha=0.009625013743012712,
                                 criterion='squared_error',
                                 dropout_rate=0.1897783294507234,
                                 learning_rate=0.015420772490455037,
                                 max_features='auto', max_leaf_nodes=14,
                                 min_impurity_decrease=5.869825897765074e-07,
                                 min_samples_leaf=13, min_samples_split=20,
                                 min_weight_fraction_leaf=0.29880213170319914,
                                 n_estimators=430, random_state=123,
                                 subsample=0.9101431135071837,
                                 validation_fraction=0.9964423942006735)

IBS: 0.229


In [155]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [156]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [157]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-15 16:51:03,020] A new study created in memory with name: no-name-c16cb83d-ab68-4766-a594-ec37963197b2


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.7131782945736435
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.49809885931558934
Fold 5 C-index: 0.5107296137339056
[I 2024-04-15 16:51:04,418] Trial 0 finished with value: 0.5698891894151084 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.5698891894151084.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.7093023255813954
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.49809885931558934
Fold 5 C-index: 0.4978540772532189
[I 2024-04-15 16:51:15,538] Trial 1 finished with value: 0.5665388883205213 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.5698891894151084.
Fold 1 C-index: 0.5577689243027888
Fold 2 C-index: 0.7325581395348837
Fold 3 C-index: 0.6085106382978723
Fol

Fold 1 C-index: 0.5537848605577689
Fold 2 C-index: 0.7364341085271318
Fold 3 C-index: 0.6382978723404256
Fold 4 C-index: 0.4866920152091255
Fold 5 C-index: 0.5793991416309013
[I 2024-04-15 16:52:54,162] Trial 19 finished with value: 0.5989215996530706 and parameters: {'subsample': 0.34314044045637715, 'dropout_rate': 0.9952300589118265, 'n_estimators': 248, 'learning_rate': 0.08042171957478578}. Best is trial 18 with value: 0.6425425582894422.
Fold 1 C-index: 0.5418326693227091
Fold 2 C-index: 0.7286821705426356
Fold 3 C-index: 0.6340425531914894
Fold 4 C-index: 0.532319391634981
Fold 5 C-index: 0.6180257510729614
[I 2024-04-15 16:52:55,326] Trial 20 finished with value: 0.6109805071529554 and parameters: {'subsample': 0.26117394341526556, 'dropout_rate': 0.8166141686494631, 'n_estimators': 112, 'learning_rate': 0.06551866052750378}. Best is trial 18 with value: 0.6425425582894422.
Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.6680851063829787


Fold 1 C-index: 0.5737051792828686
Fold 2 C-index: 0.7325581395348837
Fold 3 C-index: 0.7319148936170212
Fold 4 C-index: 0.5399239543726235
Fold 5 C-index: 0.648068669527897
[I 2024-04-15 16:54:57,174] Trial 38 finished with value: 0.6452341672670587 and parameters: {'subsample': 0.1004921614604077, 'dropout_rate': 0.9230355657351519, 'n_estimators': 427, 'learning_rate': 0.06351044805702724}. Best is trial 38 with value: 0.6452341672670587.
Fold 1 C-index: 0.5537848605577689
Fold 2 C-index: 0.7286821705426356
Fold 3 C-index: 0.6638297872340425
Fold 4 C-index: 0.5209125475285171
Fold 5 C-index: 0.6094420600858369
[I 2024-04-15 16:54:59,063] Trial 39 finished with value: 0.6153302851897602 and parameters: {'subsample': 0.2895346602735268, 'dropout_rate': 0.9803625318253736, 'n_estimators': 145, 'learning_rate': 0.0574622782928275}. Best is trial 38 with value: 0.6452341672670587.
Fold 1 C-index: 0.5577689243027888
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.6
Fold 4 C-index: 0.

...

Fold 2 IBS: 0.22173151919127074
Fold 3 IBS: 0.2240083997893821
Fold 4 IBS: 0.2464359496761848
Fold 5 IBS: 0.2271016662961729
[I 2024-04-15 17:03:26,007] Trial 19 finished with value: 0.2328307659078427 and parameters: {'subsample': 0.16667191735421055, 'dropout_rate': 0.994814822896155, 'n_estimators': 132, 'learning_rate': 0.016543957974820042}. Best is trial 12 with value: 0.23063622321443536.
Fold 1 IBS: 0.24582901698106285
Fold 2 IBS: 0.2225691567522993
Fold 3 IBS: 0.22660865453347748
Fold 4 IBS: 0.24909665057919317
Fold 5 IBS: 0.2278922684637406
[I 2024-04-15 17:03:26,636] Trial 20 finished with value: 0.23439914946195467 and parameters: {'subsample': 0.38161734041406226, 'dropout_rate': 0.5777838682216713, 'n_estimators': 54, 'learning_rate': 0.03973446561107324}. Best is trial 12 with value: 0.23063622321443536.
Fold 1 IBS: 0.24634790271407495
Fold 2 IBS: 0.2298528724561472
Fold 3 IBS: 0.22670230474026679
Fold 4 IBS: 0.24295667622798126
Fold 5 IBS: 0.2285820944108233
[I 2024-04-

Fold 2 IBS: 0.20902953968424892
Fold 3 IBS: 0.2176051201013263
Fold 4 IBS: 0.2734109979240352
Fold 5 IBS: 0.23255560033240433
[I 2024-04-15 17:04:11,381] Trial 38 finished with value: 0.23823703640240734 and parameters: {'subsample': 0.28142644997062893, 'dropout_rate': 0.26191278993485095, 'n_estimators': 253, 'learning_rate': 0.028968935059673087}. Best is trial 23 with value: 0.23052774846134477.
Fold 1 IBS: 0.27956409436548896
Fold 2 IBS: 0.2081148213434298
Fold 3 IBS: 0.21550561997048281
Fold 4 IBS: 0.30174781335524775
Fold 5 IBS: 0.2521330714304219
[I 2024-04-15 17:04:13,022] Trial 39 finished with value: 0.25141308409301427 and parameters: {'subsample': 0.19853845181024166, 'dropout_rate': 0.8550264503945254, 'n_estimators': 145, 'learning_rate': 0.09403098424740443}. Best is trial 23 with value: 0.23052774846134477.
Fold 1 IBS: 0.26770221085682805
Fold 2 IBS: 0.22837089778981953
Fold 3 IBS: 0.2288833433633006
Fold 4 IBS: 0.3030990070849948
Fold 5 IBS: 0.254317418785605
[I 2024-

Fold 2 IBS: 0.21067485967648128
Fold 3 IBS: 0.21763914316320562
Fold 4 IBS: 0.2712255729662503
Fold 5 IBS: 0.2311915839344514
[I 2024-04-15 17:05:15,745] Trial 57 finished with value: 0.23700587800409362 and parameters: {'subsample': 0.3042256220165622, 'dropout_rate': 0.9501334595910584, 'n_estimators': 441, 'learning_rate': 0.014668471268377385}. Best is trial 55 with value: 0.22815934505389251.
Fold 1 IBS: 0.2742477520549114
Fold 2 IBS: 0.2351969031539527
Fold 3 IBS: 0.23614663779050837
Fold 4 IBS: 0.31875757966455087
Fold 5 IBS: 0.2648770528469447
[I 2024-04-15 17:05:23,580] Trial 58 finished with value: 0.2658451851021736 and parameters: {'subsample': 0.6810537336561748, 'dropout_rate': 0.8338894164672775, 'n_estimators': 498, 'learning_rate': 0.020347294424858286}. Best is trial 55 with value: 0.22815934505389251.
Fold 1 IBS: 0.26015314902761866
Fold 2 IBS: 0.19334674882444652
Fold 3 IBS: 0.21209235293552248
Fold 4 IBS: 0.27679842900506585
Fold 5 IBS: 0.22618148220055354
[I 2024-

Fold 2 IBS: 0.20129330865003595
Fold 3 IBS: 0.20362472919472707
Fold 4 IBS: 0.2626098142394568
Fold 5 IBS: 0.2234413565259519
[I 2024-04-15 17:06:52,878] Trial 76 finished with value: 0.22856004141899217 and parameters: {'subsample': 0.12716991111328405, 'dropout_rate': 0.8819176940152852, 'n_estimators': 221, 'learning_rate': 0.034294132126290275}. Best is trial 55 with value: 0.22815934505389251.
Fold 1 IBS: 0.24827865332853621
Fold 2 IBS: 0.2038933620779173
Fold 3 IBS: 0.19837888063049405
Fold 4 IBS: 0.26528696281097486
Fold 5 IBS: 0.21959816902304333
[I 2024-04-15 17:06:55,351] Trial 77 finished with value: 0.22708720557419318 and parameters: {'subsample': 0.10014056994120417, 'dropout_rate': 0.9142637089723192, 'n_estimators': 220, 'learning_rate': 0.03412951382964767}. Best is trial 77 with value: 0.22708720557419318.
Fold 1 IBS: 0.2581570605057351
Fold 2 IBS: 0.2083035476250874
Fold 3 IBS: 0.21537598020680318
Fold 4 IBS: 0.2744742401793649
Fold 5 IBS: 0.22772230289799217
[I 2024

Fold 2 IBS: 0.20016155565967836
Fold 3 IBS: 0.20218202531884416
Fold 4 IBS: 0.275671025771449
Fold 5 IBS: 0.22901703205885907
[I 2024-04-15 17:07:30,793] Trial 95 finished with value: 0.23390613361156304 and parameters: {'subsample': 0.15168078965598952, 'dropout_rate': 0.9535660044358101, 'n_estimators': 218, 'learning_rate': 0.04493719007093705}. Best is trial 77 with value: 0.22708720557419318.
Fold 1 IBS: 0.2514998144114763
Fold 2 IBS: 0.20245881264210183
Fold 3 IBS: 0.1961952401625281
Fold 4 IBS: 0.26267543656746617
Fold 5 IBS: 0.2232416912945307
[I 2024-04-15 17:07:32,653] Trial 96 finished with value: 0.22721419901562062 and parameters: {'subsample': 0.13212499627568827, 'dropout_rate': 0.9975956966964254, 'n_estimators': 180, 'learning_rate': 0.042065011804907855}. Best is trial 77 with value: 0.22708720557419318.
Fold 1 IBS: 0.25603551368208766
Fold 2 IBS: 0.19900110185711675
Fold 3 IBS: 0.19314049792382357
Fold 4 IBS: 0.2681421752303816
Fold 5 IBS: 0.224723580690284
[I 2024-0

In [158]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [159]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.647
train_ibs:  0.227


#### Test

In [160]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [161]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.8053356883307206,
                                              learning_rate=0.0546539002707277,
                                              n_estimators=22, random_state=123,
                                              subsample=0.10051745578141419)

C-index score: 0.559


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.9142637089723192,
                                              learning_rate=0.03412951382964767,
                                              n_estimators=220,
                                              random_state=123,
                                              subsample=0.10014056994120417)

IBS: 0.227


In [162]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

# Results

In [163]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.845,1.0
ExtraSurvivalTrees,0.837,2.0
GradientBoosting,0.711,3.0
CoxElastic,0.705,4.0
CoxPH,0.703,5.0
CoxLasso,0.701,6.0
ComponentwiseGradientBoosting,0.647,7.0
CoxRidge,0.632,8.0


In [164]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
CoxPH,0.199,2.0
CoxLasso,0.199,2.0
CoxElastic,0.199,2.0
Randomsurvivalforest,0.205,4.0
ExtraSurvivalTrees,0.207,5.0
ComponentwiseGradientBoosting,0.227,6.0
GradientBoosting,0.234,7.0
CoxRidge,0.236,8.0


In [165]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
ComponentwiseGradientBoosting,0.559,1.0
CoxRidge,0.531,2.0
CoxPH,0.528,4.0
CoxLasso,0.528,4.0
CoxElastic,0.528,4.0
Randomsurvivalforest,0.521,6.0
GradientBoosting,0.517,7.0
ExtraSurvivalTrees,0.509,8.0


In [166]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
ComponentwiseGradientBoosting,0.227,1.0
CoxRidge,0.229,2.5
GradientBoosting,0.229,2.5
Randomsurvivalforest,0.248,4.0
ExtraSurvivalTrees,0.259,5.0
CoxLasso,0.308,6.5
CoxElastic,0.308,6.5
CoxPH,0.309,8.0


In [169]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = 'path_to_your_folder/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d2/dfs/minmax/plsr/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d2_dfs_minmax_plsr_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [170]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-15
